# Foley Pipeline (Fully Local In-Notebook)

This notebook runs the full pipeline locally with no external inference endpoints.

- Perception model runs in-notebook from Hugging Face
- Planner/refiner model runs in-notebook from Hugging Face
- AudioLDM2 and CLAP run in-notebook with stage-aware offloading

In [85]:
# Optional one-time install in your active environment
# %pip install -r ../server/requirements.txt

In [86]:
import gc
import json
import re
import cv2
import numpy as np
import torch
import scipy.io.wavfile
import librosa

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

from diffusers import AudioLDM2Pipeline
from pydub import AudioSegment

try:
    from moviepy import VideoFileClip, AudioFileClip
except Exception:
    from moviepy.editor import VideoFileClip, AudioFileClip

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Blip2ForConditionalGeneration,
    Blip2Processor,
    ClapModel,
    ClapProcessor,
)

In [87]:
@dataclass
class PipelineConfig:
    video_path: str = '../input_video.mp4'
    output_video_path: str = '../output_foley_video_local.mp4'
    generated_audio_dir: str = './generated_audio'

    # Fully local in-notebook models
    perception_model_name: str = 'Salesforce/blip2-opt-2.7b'
    planner_model_name: str = 'google/flan-t5-large'
    audio_model_name: str = 'cvssp/audioldm2'
    clap_model_name: str = 'laion/larger_clap_general'

    # Persistent local cache (prevents re-download every run)
    cache_dir: Optional[str] = '../.hf-cache'

    # Stage placement + offloading controls
    perception_device: str = ''
    planner_device: str = ''
    generation_device: str = ''
    evaluation_device: str = ''

    offload_perception_after_use: bool = True
    offload_planner_after_use: bool = True
    offload_audio_after_generation: bool = True
    offload_clap_after_evaluation: bool = True

    # Pipeline behavior
    keyframe_threshold: float = 15.0
    max_retries: int = 2
    clap_threshold: float = 0.50
    wav_sample_rate: int = 16000
    eval_sample_rate: int = 48000

    # Planner controls
    planner_max_new_tokens: int = 384
    planner_temperature: float = 0.0
    planner_attempts: int = 4
    fallback_to_vlm_plan: bool = True
    fallback_max_events: int = 12

    # Debugging and stage-contract checks
    log_stage_outputs: bool = True
    debug_dir: str = './pipeline_debug'
    strict_stage_validation: bool = True


def pick_device(prefer_gpu: bool = True) -> str:
    if prefer_gpu and torch.cuda.is_available():
        return 'cuda'
    if prefer_gpu and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


config = PipelineConfig()
config.perception_device = pick_device(prefer_gpu=True)
config.planner_device = pick_device(prefer_gpu=True)
config.generation_device = pick_device(prefer_gpu=True)
config.evaluation_device = pick_device(prefer_gpu=True)

Path(config.generated_audio_dir).mkdir(parents=True, exist_ok=True)
Path(config.cache_dir).mkdir(parents=True, exist_ok=True)
Path(config.debug_dir).mkdir(parents=True, exist_ok=True)

print('Selected devices:')
print('  perception:', config.perception_device)
print('  planner   :', config.planner_device)
print('  generation:', config.generation_device)
print('  evaluation:', config.evaluation_device)
print('HF cache dir:', Path(config.cache_dir).resolve())
print('Library versions:', f'diffusers={__import__("diffusers").__version__}', f'transformers={__import__("transformers").__version__}', f'torch={torch.__version__}')


Selected devices:
  perception: mps
  planner   : mps
  generation: mps
  evaluation: mps
HF cache dir: /Users/nisaljinadasa/Documents/Coding/GenAI---Music-in-video/.hf-cache
Library versions: diffusers=0.37.1 transformers=5.5.0 torch=2.11.0


In [88]:
def _clear_device_cache() -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def _print_device_mem(label: str, device: str) -> None:
    if device == 'cuda' and torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f'[{label}] device=cuda allocated={allocated:.2f} GB reserved={reserved:.2f} GB')
        return
    print(f'[{label}] device={device}')


def _append_stage_log(cfg: 'PipelineConfig', stage: str, content: str) -> None:
    if not getattr(cfg, 'log_stage_outputs', False):
        return
    log_dir = Path(cfg.debug_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f'{stage}.log'
    with log_path.open('a', encoding='utf-8') as f:
        f.write(content.rstrip() + '\n')
        f.write('-' * 80 + '\n')


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


def _normalize_json_like(text: str) -> str:
    t = text.strip()
    if not t:
        return t

    if t.startswith('```'):
        t = t.strip('`')
        if t.startswith('json'):
            t = t[4:].strip()

    if '{' not in t and '}' not in t and '"data"' in t:
        return '{' + t + '}'

    return t


def _extract_first_json(text: str) -> Dict[str, Any]:
    normalized = _normalize_json_like(text)

    for pattern in (r'\{.*\}', r'\[.*\]'):
        m = re.search(pattern, normalized, flags=re.DOTALL)
        if not m:
            continue

        candidate = m.group(0)
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            repaired = candidate.replace("'", '"')
            repaired = re.sub(r',\s*([}\]])', r'\1', repaired)
            parsed = json.loads(repaired)

        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, list):
            if parsed and all(isinstance(item, dict) for item in parsed):
                return {'data': parsed}
            break

    raise ValueError(f'No usable JSON object found in planner output: {text[:400]}')


class LocalModelRegistry:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

        self.perception_processor: Optional[Blip2Processor] = None
        self.perception_model: Optional[Blip2ForConditionalGeneration] = None

        self.planner_tokenizer: Optional[AutoTokenizer] = None
        self.planner_model: Optional[AutoModelForSeq2SeqLM] = None

        self.audio_pipe: Optional[AudioLDM2Pipeline] = None
        self.clap_model: Optional[ClapModel] = None
        self.clap_processor: Optional[ClapProcessor] = None

    def _dtype_for(self, device: str) -> torch.dtype:
        return torch.float16 if device in {'cuda', 'mps'} else torch.float32

    def load_perception(self, device: str) -> None:
        if self.perception_processor is None or self.perception_model is None:
            print(f'[models] Loading perception model: {self.cfg.perception_model_name}')
            self.perception_processor = Blip2Processor.from_pretrained(
                self.cfg.perception_model_name,
                cache_dir=self.cfg.cache_dir,
            )
            self.perception_model = Blip2ForConditionalGeneration.from_pretrained(
                self.cfg.perception_model_name,
                dtype=self._dtype_for(device),
                cache_dir=self.cfg.cache_dir,
            )
        self.perception_model.to(device)
        _print_device_mem('perception-loaded', device)

    def unload_perception(self) -> None:
        if self.perception_model is None:
            return
        self.perception_model.to('cpu')
        gc.collect()
        _clear_device_cache()
        _print_device_mem('perception-offloaded', 'cpu')

    def load_planner(self, device: str) -> None:
        if self.planner_tokenizer is None or self.planner_model is None:
            print(f'[models] Loading planner model: {self.cfg.planner_model_name}')
            self.planner_tokenizer = AutoTokenizer.from_pretrained(
                self.cfg.planner_model_name,
                cache_dir=self.cfg.cache_dir,
            )
            self.planner_model = AutoModelForSeq2SeqLM.from_pretrained(
                self.cfg.planner_model_name,
                dtype=self._dtype_for(device),
                cache_dir=self.cfg.cache_dir,
            )
        self.planner_model.to(device)
        _print_device_mem('planner-loaded', device)

    def unload_planner(self) -> None:
        if self.planner_model is None:
            return
        self.planner_model.to('cpu')
        gc.collect()
        _clear_device_cache()
        _print_device_mem('planner-offloaded', 'cpu')

    def load_audio_pipe(self, device: str) -> None:
        if self.audio_pipe is None:
            print(f'[models] Loading audio model: {self.cfg.audio_model_name}')
            self.audio_pipe = AudioLDM2Pipeline.from_pretrained(
                self.cfg.audio_model_name,
                torch_dtype=self._dtype_for(device),
                cache_dir=self.cfg.cache_dir,
            )

        # Compatibility fix for newer transformers where AudioLDM2 may load GPT2Model
        # instead of a generation-capable causal LM.
        lm = getattr(self.audio_pipe, 'language_model', None)
        needs_fix = (
            lm is not None
            and not hasattr(lm, '_update_model_kwargs_for_generation')
        )
        if needs_fix:
            print('[models] Re-loading AudioLDM2 language_model as AutoModelForCausalLM for compatibility')
            fixed_lm = AutoModelForCausalLM.from_pretrained(
                self.cfg.audio_model_name,
                subfolder='language_model',
                torch_dtype=self._dtype_for(device),
                cache_dir=self.cfg.cache_dir,
            )
            if hasattr(self.audio_pipe, 'register_modules'):
                self.audio_pipe.register_modules(language_model=fixed_lm)
            else:
                self.audio_pipe.language_model = fixed_lm

        self.audio_pipe.to(device)
        _print_device_mem('audio-loaded', device)

    def unload_audio_pipe(self) -> None:
        if self.audio_pipe is None:
            return
        self.audio_pipe.to('cpu')
        gc.collect()
        _clear_device_cache()
        _print_device_mem('audio-offloaded', 'cpu')

    def load_clap(self, device: str) -> None:
        if self.clap_model is None or self.clap_processor is None:
            print(f'[models] Loading CLAP model: {self.cfg.clap_model_name}')
            self.clap_model = ClapModel.from_pretrained(
                self.cfg.clap_model_name,
                cache_dir=self.cfg.cache_dir,
            )
            self.clap_processor = ClapProcessor.from_pretrained(
                self.cfg.clap_model_name,
                cache_dir=self.cfg.cache_dir,
            )
        self.clap_model.to(device)
        _print_device_mem('clap-loaded', device)

    def unload_clap(self) -> None:
        if self.clap_model is None:
            return
        self.clap_model.to('cpu')
        gc.collect()
        _clear_device_cache()
        _print_device_mem('clap-offloaded', 'cpu')


In [89]:
@dataclass
class AudioEvent:
    timestamp_sec: float
    duration_sec: float
    original_prompt: str
    refined_prompt: str = ''
    audio_file_path: Optional[str] = None
    similarity_score: float = 0.0


class PerceptionNode:
    def __init__(self, cfg: PipelineConfig, registry: LocalModelRegistry):
        self.cfg = cfg
        self.registry = registry

    def extract_keyframes(self, video_path: str) -> List[Dict[str, Any]]:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames: List[Dict[str, Any]] = []

        success, prev_frame = cap.read()
        if not success:
            cap.release()
            return frames

        frames.append({'timestamp': 0.0, 'frame': prev_frame})
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

        idx = 1
        while True:
            success, curr_frame = cap.read()
            if not success:
                break
            curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
            mean_diff = np.mean(cv2.absdiff(curr_gray, prev_gray))
            if mean_diff > self.cfg.keyframe_threshold:
                frames.append({'timestamp': round(idx / fps, 2), 'frame': curr_frame})
                prev_gray = curr_gray
            idx += 1

        cap.release()
        print(f'[Perception] Extracted {len(frames)} keyframes')
        return frames

    def analyze_video(self, video_path: str) -> str:
        keyframes = self.extract_keyframes(video_path)
        if not keyframes:
            raise RuntimeError('No frames extracted from video')

        self.registry.load_perception(self.cfg.perception_device)

        lines: List[str] = []
        for frame_info in keyframes:
            ts = frame_info['timestamp']
            bgr = frame_info['frame']
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            prompt = (
                'Describe only sound-relevant events and materials in this frame for Foley design. '
                'Keep it concise.'
            )
            inputs = self.registry.perception_processor(
                images=rgb,
                text=prompt,
                return_tensors='pt',
            )
            inputs = {
                k: v.to(self.cfg.perception_device) if hasattr(v, 'to') else v
                for k, v in inputs.items()
            }

            with torch.no_grad():
                generated_ids = self.registry.perception_model.generate(
                    **inputs,
                    max_new_tokens=80,
                    do_sample=False,
                )
            caption = self.registry.perception_processor.decode(
                generated_ids[0], skip_special_tokens=True
            ).strip()
            lines.append(f'[{ts}s] {caption}')

        if self.cfg.offload_perception_after_use:
            self.registry.unload_perception()

        vlm_log = '\n'.join(lines)
        _require(bool(vlm_log.strip()), 'Perception output is empty')
        _append_stage_log(self.cfg, '01_perception_vlm_log', vlm_log)
        print('[Perception] VLM log generated locally')
        return vlm_log


class PlannerNode:
    def __init__(self, cfg: PipelineConfig, registry: LocalModelRegistry):
        self.cfg = cfg
        self.registry = registry

    def _generate_text(self, prompt: str, max_new_tokens: Optional[int] = None) -> str:
        self.registry.load_planner(self.cfg.planner_device)

        tokenizer = self.registry.planner_tokenizer
        model = self.registry.planner_model

        tok = tokenizer(prompt, return_tensors='pt', truncation=True)
        tok = {
            k: v.to(self.cfg.planner_device) if hasattr(v, 'to') else v
            for k, v in tok.items()
        }

        with torch.no_grad():
            out = model.generate(
                **tok,
                max_new_tokens=max_new_tokens or self.cfg.planner_max_new_tokens,
                do_sample=self.cfg.planner_temperature > 0,
                temperature=self.cfg.planner_temperature,
            )

        text = tokenizer.decode(out[0], skip_special_tokens=True).strip()

        if self.cfg.offload_planner_after_use:
            self.registry.unload_planner()

        return text

    def _events_from_parsed(self, parsed: Dict[str, Any]) -> List[AudioEvent]:
        events: List[AudioEvent] = []
        for e in parsed.get('data', []):
            if not isinstance(e, dict):
                continue
            if 'timestamp_sec' not in e or 'original_prompt' not in e:
                continue
            try:
                ts = float(e['timestamp_sec'])
            except Exception:
                continue
            dur = float(e.get('duration_sec', 2.0) or 2.0)
            if dur <= 0:
                dur = 2.0
            prompt = str(e['original_prompt']).strip()
            if not prompt:
                continue
            events.append(
                AudioEvent(
                    timestamp_sec=max(0.0, ts),
                    duration_sec=dur,
                    original_prompt=prompt,
                )
            )

        events.sort(key=lambda x: x.timestamp_sec)
        return events

    def _fallback_events_from_vlm_log(self, vlm_log: str) -> List[AudioEvent]:
        pattern = re.compile(r'^\[(\d+(?:\.\d+)?)s\]\s*(.+?)\s*$')
        events: List[AudioEvent] = []
        seen = set()

        for line in vlm_log.splitlines():
            m = pattern.match(line.strip())
            if not m:
                continue
            ts = float(m.group(1))
            desc = re.sub(r'\s+', ' ', m.group(2)).strip()
            if not desc:
                continue
            key = (round(ts, 2), desc.lower())
            if key in seen:
                continue
            seen.add(key)
            events.append(
                AudioEvent(
                    timestamp_sec=max(0.0, ts),
                    duration_sec=2.0,
                    original_prompt=desc,
                )
            )
            if len(events) >= self.cfg.fallback_max_events:
                break

        events.sort(key=lambda x: x.timestamp_sec)
        return events

    def create_audio_plan(self, vlm_log: str) -> List[AudioEvent]:
        _require(bool(vlm_log.strip()), 'Planner input vlm_log is empty')

        base_prompt = (
            'Return ONLY strict JSON object. No extra words. '
            'Schema: {"data":[{"timestamp_sec":number,"duration_sec":number,"original_prompt":string}]}. '
            'Each event must contain concrete numeric timestamp_sec and duration_sec values. '
            'Use acoustic wording only and chronological ordering. '
            f'Video log:\n{vlm_log}'
        )

        attempts = max(1, int(self.cfg.planner_attempts))
        last_raw = ''
        for attempt in range(1, attempts + 1):
            prompt = base_prompt
            if attempt > 1:
                prompt += (
                    '\nPrevious output was invalid. '
                    'Do not output schema templates like timestamp:number. '
                    'Output one JSON object only with top-level key "data" and real values.'
                )

            raw = self._generate_text(prompt, max_new_tokens=self.cfg.planner_max_new_tokens)
            last_raw = raw
            _append_stage_log(
                self.cfg,
                '02_planner_attempts',
                f'ATTEMPT {attempt}/{attempts}\nPROMPT:\n{prompt}\n\nRAW:\n{raw}'
            )

            try:
                parsed = _extract_first_json(raw)
            except ValueError as exc:
                _append_stage_log(self.cfg, '02_planner_attempts', f'PARSE_ERROR: {exc}')
                print(f'[Planner] Parse failed on attempt {attempt}/{attempts}. Retrying...')
                continue

            events = self._events_from_parsed(parsed)
            if events:
                _append_stage_log(
                    self.cfg,
                    '03_planner_events',
                    json.dumps([e.__dict__ for e in events], indent=2)
                )
                print(f'[Planner] Planned {len(events)} events')
                return events

            print(f'[Planner] No valid events on attempt {attempt}/{attempts}. Retrying...')

        if self.cfg.fallback_to_vlm_plan:
            fallback = self._fallback_events_from_vlm_log(vlm_log)
            if fallback:
                _append_stage_log(
                    self.cfg,
                    '03_planner_events',
                    'FALLBACK_FROM_VLM_LOG\n' + json.dumps([e.__dict__ for e in fallback], indent=2)
                )
                print(f'[Planner] Fallback plan generated from VLM log ({len(fallback)} events)')
                return fallback

        raise ValueError(
            f'Planner returned no valid events after {attempts} attempts. '
            f'Last raw output: {last_raw[:400]}'
        )

    def refine_prompt(self, failed_prompt: str, score: float) -> str:
        prompt = (
            f"Rewrite for better Foley synthesis. Current prompt: {failed_prompt}. "
            f"Score: {score:.2f}. Return one improved prompt only."
        )
        refined = self._generate_text(prompt, max_new_tokens=96).strip()
        if not refined:
            refined = failed_prompt
        _append_stage_log(
            self.cfg,
            '05_refinements',
            f'INPUT_PROMPT: {failed_prompt}\nSCORE: {score:.4f}\nREFINED: {refined}'
        )
        return refined


class ExecutionNode:
    def __init__(self, cfg: PipelineConfig, registry: LocalModelRegistry):
        self.cfg = cfg
        self.registry = registry

    def generate_audio(self, prompt: str, timestamp: float, duration: float, attempt: int) -> str:
        _require(bool(str(prompt).strip()), f'Execution received empty prompt at t={timestamp}')
        if duration <= 0:
            duration = 2.0

        self.registry.load_audio_pipe(self.cfg.generation_device)

        enhanced_prompt = (
            f'{prompt}, high quality Foley sound effect, cinematic, crisp, detailed, 48kHz'
        )
        negative_prompt = (
            'music, speech, human voice, background noise, static, muffled, noisy, electronic, digital'
        )

        audio = self.registry.audio_pipe(
            prompt=enhanced_prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=150,
            guidance_scale=3.5,
            audio_length_in_s=duration,
        ).audios[0]

        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val
        audio_int16 = (audio * 32767).astype(np.int16)

        out_path = Path(self.cfg.generated_audio_dir) / f'event_{timestamp}_v{attempt}.wav'
        scipy.io.wavfile.write(str(out_path), rate=self.cfg.wav_sample_rate, data=audio_int16)
        _require(out_path.exists(), f'Execution failed to write audio file: {out_path}')

        _append_stage_log(
            self.cfg,
            '04_execution',
            f'timestamp={timestamp:.2f} attempt={attempt} duration={duration:.2f}\n'
            f'prompt={prompt}\noutput={out_path}'
        )
        print(f'[Execution] Generated {out_path}')

        if self.cfg.offload_audio_after_generation:
            self.registry.unload_audio_pipe()

        return str(out_path)


class VerificationNode:
    def __init__(self, cfg: PipelineConfig, registry: LocalModelRegistry):
        self.cfg = cfg
        self.registry = registry

    def evaluate(self, prompt: str, audio_path: str) -> float:
        _require(Path(audio_path).exists(), f'Verification input audio does not exist: {audio_path}')

        self.registry.load_clap(self.cfg.evaluation_device)

        audio_array, _ = librosa.load(audio_path, sr=self.cfg.eval_sample_rate)
        try:
            inputs = self.registry.clap_processor(
                text=[prompt],
                audio=audio_array,
                return_tensors='pt',
                padding=True,
                sampling_rate=self.cfg.eval_sample_rate,
            )
        except (TypeError, ValueError):
            # Backward compatibility with older transformers/processor signatures.
            inputs = self.registry.clap_processor(
                text=[prompt],
                audios=audio_array,
                return_tensors='pt',
                padding=True,
                sampling_rate=self.cfg.eval_sample_rate,
            )
        inputs = {k: v.to(self.cfg.evaluation_device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.registry.clap_model(**inputs)
            score = outputs.logits_per_audio.item() / 100.0

        if not np.isfinite(score):
            score = -1.0

        _append_stage_log(
            self.cfg,
            '06_verification',
            f"audio={audio_path}\nprompt={prompt}\nscore={score:.6f}"
        )
        print(f"[Verification] Score={score:.2f} for prompt='{prompt[:45]}...'")

        if self.cfg.offload_clap_after_evaluation:
            self.registry.unload_clap()

        return float(score)


class FoleyOrchestratorLocal:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.registry = LocalModelRegistry(cfg)
        self.perception = PerceptionNode(cfg, self.registry)
        self.planner = PlannerNode(cfg, self.registry)
        self.execution = ExecutionNode(cfg, self.registry)
        self.verification = VerificationNode(cfg, self.registry)

    def stitch_audio_to_video(self, video_path: str, events: List[AudioEvent], output_path: str) -> None:
        print('[Stitcher] Building final mix')
        video_clip = VideoFileClip(video_path)
        base_audio = AudioSegment.silent(duration=int(video_clip.duration * 1000))

        for event in events:
            if event.audio_file_path and Path(event.audio_file_path).exists():
                foley_clip = AudioSegment.from_wav(event.audio_file_path)
                position_ms = int(event.timestamp_sec * 1000)
                base_audio = base_audio.overlay(foley_clip, position=position_ms)

        temp_audio = Path(self.cfg.generated_audio_dir) / 'temp_mixed_audio.wav'
        base_audio.export(str(temp_audio), format='wav')

        final_audio_clip = AudioFileClip(str(temp_audio))
        final_video = video_clip.with_audio(final_audio_clip)
        final_video.write_videofile(output_path, codec='libx264', audio_codec='aac')

        temp_audio.unlink(missing_ok=True)
        _append_stage_log(
            self.cfg,
            '07_stitch',
            f'output_video={output_path}\nnum_events={len(events)}'
        )
        print(f'[Stitcher] Final video saved: {output_path}')

    def run_pipeline(self) -> List[AudioEvent]:
        print(f'--- Starting local Foley pipeline for {self.cfg.video_path} ---')

        vlm_log = self.perception.analyze_video(self.cfg.video_path)
        _require(bool(vlm_log.strip()), 'Perception produced empty vlm_log')

        audio_plan = self.planner.create_audio_plan(vlm_log)
        _require(len(audio_plan) > 0, 'Planner produced an empty audio plan')

        final_events: List[AudioEvent] = []
        for event in audio_plan:
            current_prompt = event.original_prompt
            success = False
            audio_path = ''

            for attempt in range(1, self.cfg.max_retries + 1):
                print(f'  -> Event {event.timestamp_sec:.2f}s | attempt {attempt}/{self.cfg.max_retries}')
                audio_path = self.execution.generate_audio(
                    current_prompt,
                    event.timestamp_sec,
                    event.duration_sec,
                    attempt,
                )
                score = self.verification.evaluate(current_prompt, audio_path)

                if score >= self.cfg.clap_threshold:
                    event.audio_file_path = audio_path
                    event.similarity_score = score
                    event.refined_prompt = current_prompt
                    success = True
                    print('  -> [Pass] Threshold met')
                    break

                current_prompt = self.planner.refine_prompt(current_prompt, score)
                print('  -> [Retry] Prompt refined')

            if not success:
                event.audio_file_path = audio_path
                event.refined_prompt = current_prompt
                print('  -> [Best Effort] Max retries reached')

            final_events.append(event)
            print('-' * 50)

        _append_stage_log(
            self.cfg,
            '08_final_events',
            json.dumps([e.__dict__ for e in final_events], indent=2)
        )
        self.stitch_audio_to_video(self.cfg.video_path, final_events, self.cfg.output_video_path)
        return final_events


In [90]:
# Stage placement presets
config.video_path = '../input_video.mp4'
config.output_video_path = '../output_foley_video_local.mp4'

# Use fastest local accelerator available (cuda -> mps -> cpu)
config.perception_device = pick_device(prefer_gpu=True)
config.planner_device = pick_device(prefer_gpu=True)
config.generation_device = pick_device(prefer_gpu=True)
config.evaluation_device = pick_device(prefer_gpu=True)

# If memory is tight, move planner/eval to CPU
# config.planner_device = 'cpu'
# config.evaluation_device = 'cpu'

config.offload_perception_after_use = True
config.offload_planner_after_use = True
config.offload_audio_after_generation = True
config.offload_clap_after_evaluation = True

# Quality/speed
config.max_retries = 2
config.clap_threshold = 0.50
config.planner_temperature = 0.0

print('Devices:')
print(' perception =', config.perception_device)
print(' planner    =', config.planner_device)
print(' generation =', config.generation_device)
print(' evaluation =', config.evaluation_device)
print('cache_dir   =', Path(config.cache_dir).resolve())


Devices:
 perception = mps
 planner    = mps
 generation = mps
 evaluation = mps
cache_dir   = /Users/nisaljinadasa/Documents/Coding/GenAI---Music-in-video/.hf-cache


In [91]:
# Warmup/download check: this cell downloads once into cache_dir if missing.
registry = LocalModelRegistry(config)
registry.load_perception(config.perception_device)
if config.offload_perception_after_use:
    registry.unload_perception()

registry.load_planner(config.planner_device)
if config.offload_planner_after_use:
    registry.unload_planner()

registry.load_audio_pipe(config.generation_device)
if config.offload_audio_after_generation:
    registry.unload_audio_pipe()

registry.load_clap(config.evaluation_device)
if config.offload_clap_after_evaluation:
    registry.unload_clap()

print('Local model warmup/download complete')
print('Models cached at:', Path(config.cache_dir).resolve())


[models] Loading perception model: Salesforce/blip2-opt-2.7b


Loading weights: 100%|██████████| 1247/1247 [00:24<00:00, 51.87it/s]


[perception-loaded] device=mps
[perception-offloaded] device=cpu
[models] Loading planner model: google/flan-t5-large


Loading weights: 100%|██████████| 558/558 [00:01<00:00, 400.61it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[planner-loaded] device=mps
[planner-offloaded] device=cpu
[models] Loading audio model: cvssp/audioldm2


...

[audio-loaded] device=mps


Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, du

[audio-offloaded] device=cpu
[models] Loading CLAP model: laion/larger_clap_general


Loading weights: 100%|██████████| 555/555 [00:00<00:00, 42919.75it/s]


[clap-loaded] device=mps
[clap-offloaded] device=cpu
Local model warmup/download complete
Models cached at: /Users/nisaljinadasa/Documents/Coding/GenAI---Music-in-video/.hf-cache


In [92]:
# Run full pipeline
orchestrator = FoleyOrchestratorLocal(config)
results = orchestrator.run_pipeline()

--- Starting local Foley pipeline for ../input_video.mp4 ---
[Perception] Extracted 7 keyframes
[models] Loading perception model: Salesforce/blip2-opt-2.7b


Loading weights: 100%|██████████| 1247/1247 [00:23<00:00, 53.34it/s]


[perception-loaded] device=mps
[perception-offloaded] device=cpu
[Perception] VLM log generated locally
[models] Loading planner model: google/flan-t5-large


Loading weights: 100%|██████████| 558/558 [00:02<00:00, 229.91it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[planner-loaded] device=mps
[planner-offloaded] device=cpu
[Planner] Parse failed on attempt 1/4. Retrying...


KeyboardInterrupt: 

In [ ]:
# Summary
for i, e in enumerate(results, start=1):
    print(
        f"{i}. t={e.timestamp_sec:.2f}s dur={e.duration_sec:.2f}s score={e.similarity_score:.2f} "
        f"file={e.audio_file_path}"
    )

## Notes

- First warmup run can take a long time due to local model downloads.
- If perception model is too heavy, switch `perception_model_name` to a smaller image-caption model.
- If planner JSON parse fails, lower `planner_temperature` to `0.0`.